In [1]:
import pandas as pd

# Path to data frame containing all WSIs with a matched rekvnr
df_path = "D:\DATA\overlapping_rekvnr.xlsx"
df_overlapping = pd.read_excel(df_path)

In [2]:
print(df_overlapping.head())

     rekvnr    modtdato team sex  alder alder gruppe    rekvdato  matantal  \
0  12200001  2012-01-02  URO   M     52        50-54  2011-12-30        12   
1  12200019  2012-01-02  URO   M     72          65+  2012-01-02         2   
2  12200014  2012-01-02  LUG   F     87          65+  2012-01-02         1   
3  12100029  2012-01-02  LUG   F     87          65+  2012-01-02         3   
4  12200016  2012-01-02  URO   M     74          65+  2012-01-02        12   

   mattype mattype tekst                                         makrotekst  \
0       11     Hist. små  Trådformet vævsstykke, 15 mm. 1 kps. /PWN,CN ;...   
1       12   Hist. store  Fedtvæv indeholdende lymfeknuder. Har samlet m...   
2       11     Hist. små  Flere trådformede vævsstykker tilsammen målend...   
3       24    Anden cyt.                Modtaget rød let grumset væske/tro    
4       11     Hist. små  Trådformet vævsstykke målende 22 mm, alt med i...   

                                          mikrotekst  \


In [3]:
# Overview of all missing values
missing_counts = df_overlapping.isnull().sum()
print("\nMissing values: \n", missing_counts)


Missing values: 
 rekvnr             0
modtdato           0
team               0
sex                0
alder              0
alder gruppe       0
rekvdato           0
matantal           0
mattype            0
mattype tekst      0
makrotekst        20
mikrotekst       323
snomed kode        0
kode fritekst      0
wsi count          0
wsi filenames      0
dtype: int64


In [4]:
# Path to SNOMED codes (all codes with code history)
snomed_path = "D:/DATA/patoSnoMed_2025-04.xlsx"

xls_snomed = pd.read_excel(snomed_path)
print(f"Columns: {xls_snomed.columns.tolist()}")

Columns: ['Art', 'SKSkode', 'DatoFra', 'DatoÆndring', 'DatoTil', 'Kodetekst', 'AGrp', 'BGrp', 'CGrp', 'DGrp', 'EGrp', 'AVal', 'BVal', 'CVal', 'DVal', 'Inklusion', 'EkstraReg', 'Tags', 'Farvekode', 'Fuldtekst']


In [5]:
# Data Frame with relevant columns
df_snomed = pd.DataFrame(xls_snomed, columns=['SKSkode', 'DatoFra', 'DatoÆndring', 'DatoTil', 'Kodetekst', 'Fuldtekst'])

print(df_snomed.head())

  SKSkode   DatoFra  DatoÆndring   DatoTil  \
0  EYYY00  19400101     19400101  20000101   
1  F00150  19400101     19400101  25000101   
2  F01050  19400101     19400101  25000101   
3  F01051  19400101     20091218  20091231   
4  F01051  20100101     20150921  20150930   

                              Kodetekst                             Fuldtekst  
0  udgået (forkert oprettet, se ÆYYY00)  udgået (forkert oprettet, se ÆYYY00)  
1                               kakeksi                               kakeksi  
2                 postoperativ tilstand                 postoperativ tilstand  
3             komplet mesorektal fascie             komplet mesorektal fascie  
4                      mesorektalt plan                      mesorektalt plan  


In [6]:
# Convert SNOMED codes to a set for fast lookup
snomed_set = set(df_snomed['SKSkode'].dropna().astype(str))

# Function to get first letters of SNOMED codes
def get_first_letters(codes):
    """ Given a list of SNOMED codes, return a list of unique first letters. """
    return {code[0] for code in codes if code}

# Get unique first letters
first_letters = get_first_letters(df_snomed['SKSkode'].dropna().unique())
print('SNOMED letters:', first_letters)


SNOMED letters: {'Æ', 'T', 'J', 'P', 'E', 'F', 'S', 'M'}


In [7]:
# Code to Find Valid SNOMED Codes in "snomed kode" column
# All valid codes (with history) in snomed_set

import re

# Function to extract unique valid snomed codes by first letter
def extract_by_letter(cell, letter, valid_codes):
    if pd.isnull(cell):
        return []
    # Split string by common separators
    parts = re.split(r"[ ,;]+", cell)
    # Keep only unique, valid codes that start with the desired letter
    codes = {part for part in parts if part in valid_codes and part.startswith(letter)}
    return list(codes) 

# Define which letters to treat separately
main_letters = ["T", "M"]
other_letters = list(first_letters - set(main_letters))

# Apply for T and M
for letter in main_letters:
    df_overlapping[letter] = df_overlapping["snomed kode"].apply(lambda x: extract_by_letter(x, letter, snomed_set))

# Apply for all other letters and combine into one "Other" column
def extract_other(cell, main_letters, valid_codes):
    if pd.isnull(cell):
        return []
    parts = re.split(r"[ ,;]+", cell)
    codes = {part for part in parts if part in valid_codes and part[0] not in main_letters}
    return list(codes)

df_overlapping["Other"] = df_overlapping["snomed kode"].apply(lambda x: extract_other(x, main_letters, snomed_set))

# Check result
print(df_overlapping.head())

     rekvnr    modtdato team sex  alder alder gruppe    rekvdato  matantal  \
0  12200001  2012-01-02  URO   M     52        50-54  2011-12-30        12   
1  12200019  2012-01-02  URO   M     72          65+  2012-01-02         2   
2  12200014  2012-01-02  LUG   F     87          65+  2012-01-02         1   
3  12100029  2012-01-02  LUG   F     87          65+  2012-01-02         3   
4  12200016  2012-01-02  URO   M     74          65+  2012-01-02        12   

   mattype mattype tekst                                         makrotekst  \
0       11     Hist. små  Trådformet vævsstykke, 15 mm. 1 kps. /PWN,CN ;...   
1       12   Hist. store  Fedtvæv indeholdende lymfeknuder. Har samlet m...   
2       11     Hist. små  Flere trådformede vævsstykker tilsammen målend...   
3       24    Anden cyt.                Modtaget rød let grumset væske/tro    
4       11     Hist. små  Trådformet vævsstykke målende 22 mm, alt med i...   

                                          mikrotekst  \


In [8]:
# Function to check for missing values or empty lists
def is_missing(x):
    try:
        if isinstance(x, list):
            return len(x) == 0
        elif isinstance (x, tuple):
            return len(x) == 0
        return pd.isnull(x)
    except Exception:
        return False

# Check column T and M
missing_mask_T = df_overlapping["T"].apply(is_missing)
missing_mask_M = df_overlapping["M"].apply(is_missing)

print("Number of missing values in T:", missing_mask_T.sum())
print("Number of missing values in M:", missing_mask_M.sum())

Number of missing values in T: 52
Number of missing values in M: 52


In [9]:
# Code to find missing T and M codes in text columns, using valid codes set

text_columns = ["makrotekst", "mikrotekst", "kode fritekst"]

def find_missing_codes(row, letter, columns, valid_codes):
    if is_missing(row[letter]):
        found = []
        for c in columns: 
            codes = extract_by_letter(row[c], letter, valid_codes)
            if codes: 
                found.extend(codes)
        return list(set(found)) if found else None
    return None

# Apply for T and M
for letter in main_letters:
    df_overlapping[f"{letter} candidates"] = df_overlapping.apply(lambda r: find_missing_codes(r, letter, text_columns, snomed_set), axis=1)

# Count rows where T and M is missing but candidates found
fixable_T_count = df_overlapping["T candidates"].notna().sum()
fixable_M_count = df_overlapping["M candidates"].notna().sum()

print("Number of fixable T rows:", fixable_T_count)
print("Number of fixable M rows:", fixable_M_count)

Number of fixable T rows: 13
Number of fixable M rows: 13


In [10]:
fixable_T = df_overlapping[df_overlapping["T candidates"].notna()]
print("\nFixable T rows: \n", fixable_T['T candidates'])

fixable_M = df_overlapping[df_overlapping["M candidates"].notna()]
print("\nFixable M rows: \n", fixable_M['M candidates'])


Fixable T rows: 
 237              [T77100]
651              [T02450]
1563             [T02424]
1731             [T77100]
3940     [T00100, T83701]
5430             [T83701]
5484             [T83701]
6346             [T74010]
8414             [T02600]
9629     [T84000, T85000]
10252    [T88000, T88225]
11202            [T02400]
14384            [T02480]
Name: T candidates, dtype: object

Fixable M rows: 
 237                      [M09450]
651                      [M09450]
1563                     [M72751]
1731             [M81403, M09450]
3940             [M80763, M807A2]
5430             [M807A2, M09400]
5484             [M807A2, M09400]
6346             [M45020, M42100]
8414                     [M48000]
9629             [M79320, M00100]
10252    [MÆ0025, M00100, M37100]
11202                    [M51570]
14384                    [M48000]
Name: M candidates, dtype: object


In [11]:
# Code to fill SNOMED codes from text columns

def fill_from_candidates(row, col, valid_codes):
    current = row[col]
    candidates = row[f"{col} candidates"]

    # Check if current is missing (None or empty list)
    if is_missing(current):
        if candidates is not None:
            # Ensure unique + valid
            return list({c for c in candidates if c in valid_codes})
        return []
    return current

for letter in main_letters: 
    df_overlapping[letter] = df_overlapping.apply(lambda r: fill_from_candidates(r, letter, snomed_set), axis=1)


In [12]:
# Check missing values again
missing_mask_T = df_overlapping["T"].apply(is_missing)
missing_mask_M = df_overlapping["M"].apply(is_missing)

print("Number of missing values in T:", missing_mask_T.sum())
print("Number of missing values in M:", missing_mask_M.sum())


Number of missing values in T: 39
Number of missing values in M: 39


In [13]:
# Duplicate Rows

# Convert lists to tuples
df_tuples = df_overlapping.map(
    lambda x: tuple(x) if isinstance(x, list) else x  
)

print("Number of duplicated rows:", df_tuples.duplicated().sum())

Number of duplicated rows: 0


In [14]:
# Duplicate values in rekvnr

# Mask for rows with duplicate rekvnr
dup_mask = df_tuples["rekvnr"].duplicated(keep=False)
df_dups = df_tuples[dup_mask]

num_dups = df_tuples["rekvnr"].duplicated().sum()
print("Number of duplicate rekvnr:", num_dups)

Number of duplicate rekvnr: 5


In [15]:
# Show rows with duplicate rekvnr
df_dups_sorted = df_dups.sort_values("rekvnr")

print(df_dups_sorted)

        rekvnr    modtdato team sex  alder alder gruppe    rekvdato  matantal  \
4373  12204567  2012-04-03  GYN   F     25        25-29  2012-04-02         2   
5480  12204567  2012-04-03  GYN   F     25        25-29  2012-04-02         2   
4438  12204648  2012-04-10  LUG   M     69          65+  2012-04-04         1   
5481  12204648  2012-04-10  LUG   M     69          65+  2012-04-04         1   
5172  12205400  2012-04-24  TMK   M     36        35-39  2012-04-24         2   
5482  12205400  2012-04-24  TMK   M     36        35-39  2012-04-24         2   
5387  12205620  2012-04-27  HUD   F     45        45-49  2012-04-25         1   
5483  12205620  2012-04-27  HUD   F     45        45-49  2012-04-25         1   
5430  12205712  2012-04-30  GYN   F     33        30-34  2012-04-26         1   
5484  12205712  2012-04-30  GYN   F     33        30-34  2012-04-26         1   

      mattype mattype tekst  ...  \
4373       12   Hist. store  ...   
5480       12   Hist. store  ...   


In [ ]:
def combine_values(series):
    """ Combine differing values in a column into a list of unique values. """
    # If all values are the same, return the single value
    uniques = series.dropna().unique()
    if len(uniques) == 1:
        return uniques[0]
    else:
        return list(uniques)

# Combine rows with same rekvnr and modtdato
df_no_dups = df_tuples.groupby(["rekvnr", "modtdato"], as_index=False).agg(combine_values)


In [ ]:
# Get results
print("Before combing duplicated rekvnr:", len(df_tuples))
print("After combining rows with duplicate rekvnr:", len(df_no_dups))

In [ ]:
# Check missing values in all columns
for col in df_no_dups.columns: 
    missing_count = df_no_dups[col].apply(is_missing).sum()
    print(f"{col}: ", missing_count)

In [ ]:
# Get number of files with missing T or M codes

# Step 1: Find rows with missing T or M
rows_with_missing = df_no_dups[df_no_dups['T'].apply(is_missing) | df_no_dups['M'].apply(is_missing)]

# Step 2: Count total number of filenames in those rows
unique_files_missing = set().union(*rows_with_missing['wsi filenames'])
print("Total number of unique files with missing T or M:", len(unique_files_missing))

# Step 3: Remove those rows from df_wsi
df_clean = df_no_dups.drop(rows_with_missing.index)


In [ ]:
# Get results
print("Before removing missing values:", len(df_no_dups))
print("After removing rows with missing T or M:", len(df_clean))

In [ ]:
# Remove candidate columns
cols_to_drop = ["T candidates", "M candidates"]
df_clean = df_clean.drop(columns=cols_to_drop)

In [ ]:
# Check correct values / formatting
print(df_clean.head(3))

In [ ]:
# Check correct formatting
def get_pattern(col):
    if col == "rekvnr": 
        return r"\d{8}"           # 8 digits
    elif col == "modtdato" or col == "rekvdato":
        return r"\d{4}-\d{2}-\d{2}"  # YYYY-MM-DD
    elif col == "team":
        return r"[A-ZÆØÅ]{3}"           # 3 uppercase letters
    elif col == "sex":
        return r"[FMfm]"               # F or M
    elif col == "alder" or col == "matantal" or col == "mattype" or col == "wsi count":
        return r"\d+"             # digits
    else:
        return None

def is_correct(value, pattern):
    if pattern is None or pd.isna(value):
        return False
    return bool(re.fullmatch(pattern, str(value).strip()))


In [ ]:
for col in df_clean.columns: 
    pattern = get_pattern(col)
    if pattern is None:
        continue
    incorrect = (~df_clean[col].apply(lambda x: is_correct(x, pattern))).sum()
    print(f"Number of {col} with incorrect formatting:", incorrect)

In [ ]:
# Show all unique values in the 'sex' column
unique_sex_values = df_clean['sex'].unique()
print(unique_sex_values)

# Optionally, count occurrences of each value
sex_counts = df_clean['sex'].value_counts(dropna=False)
print(sex_counts)


In [ ]:
# Standardize sex column to uppercase F or M
df_clean['sex'] = df_clean['sex'].apply(lambda x: str(x).upper() if pd.notna(x) else x)

# Check unique values
print(df_clean['sex'].unique())


In [ ]:
# Show all unique values in the 'team' column
unique_team_values = df_clean['team'].unique()
print(unique_team_values)

# Optionally, count occurrences of each value
team_counts = df_clean['team'].value_counts(dropna=False)
print(team_counts)


In [ ]:
for col in df_clean.columns: 
    print(f"{col}: :", type(col))

In [ ]:
# CHECK DATES

# Convert to datetime if not already
df_clean['modtdato'] = pd.to_datetime(df_clean['modtdato'], errors='coerce')
df_clean['rekvdato'] = pd.to_datetime(df_clean['rekvdato'], errors='coerce')

# Only allows dates from 2011 - 2013
mask_valid_dates = (df_clean['modtdato'].dt.year >= 2011) & (df_clean['modtdato'].dt.year <= 2013)
invalid_dates = df_clean[~mask_valid_dates]
print("Rows with modtdato outside 2011-2013:", len(invalid_dates))

# MODTDATO must be after REKVDATO
mask_modtdato_before_rekvdato = df_clean['modtdato'] < df_clean['rekvdato']
invalid_modtdato = df_clean[mask_modtdato_before_rekvdato]
print("Rows with modtdato later than rekvdato:", len(invalid_modtdato))


In [ ]:
# CHECK NUMERIC COLUMNS

# Age check
invalid_age = df_clean[(df_clean['alder'] < 0) | (df_clean['alder'] > 120)]
print("Rows with invalid age:", len(invalid_age))

# matantal and wsi count should be >= 0
invalid_matantal = df_clean[df_clean['matantal'] < 0]
invalid_wsi_count = df_clean[df_clean['wsi count'] < 0]
print("Rows with negative matantal:", len(invalid_matantal))
print("Rows with negative wsi count:", len(invalid_wsi_count))

In [30]:
print(df_clean.head())
print("\nData frame shape: ", df_clean.shape)


     rekvnr    modtdato team sex  alder alder gruppe    rekvdato  matantal  \
0  12100029  2012-01-02  LUG   F     87          65+  2012-01-02         3   
1  12102211  2012-02-09  LUG   F     36        35-39  2012-02-07         1   
2  12105184  2012-03-28  LUG   M     87          65+  2012-03-27         1   
3  12200001  2012-01-02  URO   M     52        50-54  2011-12-30        12   
4  12200014  2012-01-02  LUG   F     87          65+  2012-01-02         1   

   mattype mattype tekst                                         makrotekst  \
0       24    Anden cyt.                Modtaget rød let grumset væske/tro    
1       23         Smear                                                 []   
2       26          Urin                                                 []   
3       11     Hist. små  Trådformet vævsstykke, 15 mm. 1 kps. /PWN,CN ;...   
4       11     Hist. små  Flere trådformede vævsstykker tilsammen målend...   

                                          mikrotekst  \


In [ ]:
# Save to Excel
output_file = "D:\DATA\df_cleaned.xlsx"
df_clean.to_excel(output_file, index=False)

print(f"Saved DataFrame to {output_file}")